# Link2Shorts — Google Colab (No Termux)
Run this notebook in Google Colab. Processing happens on Colab's temporary runtime, not on your phone. Use only videos you own or are authorized to process. Free Colab availability and runtime limits apply; YouTube may block hosted download requests.

In [ ]:
!git clone --depth 1 https://github.com/mrynxgod-ops/Link2Shorts.git
%cd Link2Shorts
!pip -q install -r requirements.txt
!ffmpeg -version | head -n 1

In [ ]:
from getpass import getpass
from IPython.display import display, Markdown
import subprocess, os, zipfile
from pathlib import Path

video_url = getpass('Paste your authorized video URL (input is hidden): ').strip()
count = int(input('How many Shorts? (1–8): ').strip() or '3')
model = input('Whisper model tiny/base/small [tiny]: ').strip().lower() or 'tiny'
if not video_url.startswith(('https://','http://')):
    raise ValueError('Please enter a valid http(s) video URL.')
if count < 1 or count > 8:
    raise ValueError('Clip count must be between 1 and 8.')
if model not in {'tiny','base','small'}:
    raise ValueError('Model must be tiny, base, or small.')

cmd = ['python','clipper.py','--url',video_url,'--count',str(count),'--model',model]
print('Starting cloud processing. Keep this Colab tab open while it runs.')
subprocess.run(cmd, check=True)

In [ ]:
from google.colab import files
out = Path('output')
clips = sorted(out.glob('short_*.mp4'))
if not clips:
    raise FileNotFoundError('No clips were created. Check the error output in the previous cell.')
archive = Path('/content/link2shorts_clips.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in clips:
        z.write(p, p.name)
    manifest = out / 'manifest.json'
    if manifest.exists(): z.write(manifest, manifest.name)
print(f'Created {len(clips)} Shorts. Downloading ZIP...')
files.download(str(archive))

## If download fails
- A message such as **Sign in to confirm you're not a bot** means YouTube is blocking the cloud runtime. Colab cannot guarantee that a YouTube URL will download.
- Do not use this notebook to bypass sign-in, DRM, age, geographic, or other access restrictions.
- Try a direct media URL that is publicly accessible and that you are permitted to process, or a video file you have rights to use.
- Free Colab sessions can disconnect and have resource/time limits. This is not unlimited cloud processing.